# Inner-Node Proximity Strategies and LCA Distance

Deep regression forests often fragment the training data into many tiny leaves. When Forest-Guided Clustering uses plain terminal-node co-occurrence, that fragmentation can make the proximity matrix very sparse: many sample pairs never land in exactly the same leaf, even if their decision paths stay close for most of the tree.

This notebook compares four ways to make those distances more informative on the same regression problem: `min_samples_in_node`, `max_depth_for_proximity`, `min_node_variance`, and `DistanceRandomForestLCA`. The broader series is tracked in [issue #27](https://github.com/HelmholtzAI-Consultants-Munich/fg-clustering/issues/27), with the feature work split across [PR #39](https://github.com/HelmholtzAI-Consultants-Munich/fg-clustering/pull/39), [PR #42](https://github.com/HelmholtzAI-Consultants-Munich/fg-clustering/pull/42), [PR #43](https://github.com/HelmholtzAI-Consultants-Munich/fg-clustering/pull/43), and [PR #44](https://github.com/HelmholtzAI-Consultants-Munich/fg-clustering/pull/44).


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_regression
from sklearn.ensemble import RandomForestRegressor

from fgclustering import (
    ClusteringKMedoids,
    DistanceRandomForestLCA,
    DistanceRandomForestProximity,
)

RANDOM_STATE = 0
K = 4

X_raw, y = make_regression(
    n_samples=160,
    n_features=8,
    n_informative=5,
    noise=20.0,
    random_state=RANDOM_STATE,
)
X = pd.DataFrame(X_raw, columns=[f"feature_{i}" for i in range(X_raw.shape[1])])
sample_idx = np.arange(len(X))

regressor = RandomForestRegressor(
    n_estimators=100,
    max_depth=12,
    random_state=RANDOM_STATE,
)
regressor.fit(X, y)

variance_threshold = float(np.var(y) * 0.1)
print(f"Dataset shape: {X.shape}, variance threshold: {variance_threshold:.2f}")


def run_distance(distance_metric, label):
    distance_metric.calculate_terminals(estimator=regressor, X=X)
    clustering = ClusteringKMedoids(random_state=RANDOM_STATE)
    labels = clustering.run_clustering(
        k=K,
        distance_metric=distance_metric,
        sample_indices=sample_idx,
        random_state_subsampling=None,
        verbose=0,
    )
    counts = np.bincount(labels, minlength=K + 1)[1:]
    return {"name": label, "labels": labels, "counts": counts}


def plot_cluster_sizes(reference, candidate):
    clusters = np.arange(1, K + 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
    colors = ("#355070", "#b56576")
    for ax, result, color in zip(axes, (reference, candidate), colors):
        ax.bar(clusters, result["counts"], color=color, width=0.7)
        ax.set_title(result["name"])
        ax.set_xlabel("Cluster")
        ax.set_xticks(clusters)
        ax.grid(axis="y", alpha=0.25)
    axes[0].set_ylabel("Samples")
    fig.suptitle("Cluster-size histogram comparison")
    fig.tight_layout()
    plt.show()


def summarize(result):
    print(result["name"])
    print("cluster sizes:", dict(zip(range(1, K + 1), result["counts"])))


In [ ]:
baseline = run_distance(DistanceRandomForestProximity(), "Baseline terminal-node proximity")
summarize(baseline)
plot_cluster_sizes(baseline, baseline)


In [ ]:
min_samples_result = run_distance(
    DistanceRandomForestProximity(min_samples_in_node=20),
    "Collapsed proximity: min_samples_in_node=20",
)
summarize(min_samples_result)
plot_cluster_sizes(baseline, min_samples_result)


In [ ]:
max_depth_result = run_distance(
    DistanceRandomForestProximity(max_depth_for_proximity=4),
    "Collapsed proximity: max_depth_for_proximity=4",
)
summarize(max_depth_result)
plot_cluster_sizes(baseline, max_depth_result)


In [ ]:
variance_result = run_distance(
    DistanceRandomForestProximity(min_node_variance=variance_threshold),
    f"Collapsed proximity: min_node_variance={variance_threshold:.2f}",
)
summarize(variance_result)
plot_cluster_sizes(baseline, variance_result)


In [ ]:
lca_result = run_distance(
    DistanceRandomForestLCA(),
    "LCA distance",
)
summarize(lca_result)
plot_cluster_sizes(baseline, lca_result)


## Selection Guidance

- Pick `min_samples_in_node` when your main problem is leaf sparsity and you want a simple data-density floor.
- Pick `max_depth_for_proximity` when you want a fixed, model-structure-based coarsening rule that is easy to reason about across datasets.
- Pick `min_node_variance` when you care specifically about preserving response heterogeneity in regression forests.
- Pick `DistanceRandomForestLCA` when exact leaf co-occurrence is too brittle and you want a graded path-similarity metric instead of collapsing everything to shared ancestors first.
